In [1]:
import numpy as np
import xarray as xr

In [2]:
def eof(X,n=-1,detrend='constant',eof_in=None):
	"""Principal Component Analysis / Empirical Orthogonal Functions / SVD

		Uses Singular Value Decomposition to find the dominant modes of variability.
		The field X can be reconstructed with Y = dot(EOF,PC) + X.mean(axis=time)

		INPUTS:
			X	-- Field, shape (time x space).
			n	-- Number of modes to extract. All modes if n < 0
			detrend -- detrend with global mean ('constant')
						  or linear trend ('linear')
		    eof_in  -- If not None, compute PC by projecting eof onto X.
		OUTPUTS:
			EOF - Spatial modes of variability
			PC  - Temporal evolution of EOFs - only output if eof_in is not None
			E   - Explained value of variability
			u   - spatial modes
			s   - variances
			v   - temporal modes

    from aostools from Martin Jucker
	"""
	is_xr = False
	try:
		import xarray as xr
		if isinstance(X,xr.DataArray):
			is_xr = True
			dims = []
			for dim in X.dims[1:]:
				dims.append(X[dim])
			tdim = [X[X.dims[0]]]
			X = X.values
	except:
		pass
	import scipy.signal as sg
	# make sure we have a matrix time x space
	shpe = X.shape
	if len(shpe) > 2:
		X = X.reshape([shpe[0],np.prod(shpe[1:])])
		if eof_in is not None:
			if len(eof_in.shape) > 2:
				eof_in = eof_in.reshape([np.prod(eof_in.shape[:-1]),eof_in.shape[-1]])
			else:
				eof_in = eof_in.reshape([np.prod(eof_in.shape),1])
	# take out the time mean or trend
	X = sg.detrend(X.transpose(),type=detrend)
	if eof_in is not None:
		if eof_in.shape[-1] == X.shape[0]:
			PC =  np.matmul(eof_in, X)
			eof_norm = np.dot(eof_in.transpose(),eof_in)
			return np.dot(PC,np.linalg.inv(eof_norm))
		else:
			PC = np.matmul(eof_in.transpose(), X)
			eof_norm = np.dot(eof_in.transpose(),eof_in)
			return np.dot(PC.transpose(),np.linalg.inv(eof_norm)).transpose()
		# return sg.detrend(PC,type='constant')
	# perform SVD - v is actually V.H in X = U*S*V.H
	u,s,v = np.linalg.svd(X, full_matrices=False)
	# now, u contains the spatial, and v the temporal structures
	# s contains the variances, with the same units as the input X
	# u.shape = (space, modes(space)), v.shape = (modes(space), time)

	# get the first n modes, in physical units
	#  we can either project the data onto the principal component, X*V
	#  or multiply u*s. This is the same, as U*S*V.H*V = U*S
	if n < 0:
		n = s.shape[0]
	EOF = np.dot(u[:,:n],np.diag(s)[:n,:n])
	# time evolution is in v
	PC  = v[:n,:]
	# EOF wants \lambda = the squares of the eigenvalues,
	#  but SVD yields \gamma = \sqrt{\lambda}
	s2 = s*s
	E   = s2[:n]/sum(s2)
	# now we need to make sure we get everything into the correct shape again
	u = u[:,:n]
	s = s[:n]
	v = v.transpose()[:,:n]
	if len(shpe) > 2:
		# replace time dimension with modes at the end of the array
		newshape = list(shpe[1:])+[n]
		EOF = EOF.reshape(newshape)
		u   = u	 .reshape(newshape)
	if is_xr: # return xarray dataarrays
		mode = [('n',np.arange(1,n+1))]
		EOF = xr.DataArray(EOF,coords=dims+mode,name='EOF')
		PC  = xr.DataArray(PC,coords=tdim+mode,name='PC')
		E   = xr.DataArray(E,coords=mode,name='E')
	return EOF,PC,E,u,s,v

In [48]:
datadir = 'data/z10_eunpa/'

In [49]:
gh_fpath = datadir + "z10.era5.*.nc"

In [10]:
ilev = 1000 # Pa

def pre(ds): return ds[['var129']].sel(lat=slice(-20,-90)).sel(plev=ilev)

In [16]:
ep_z = xr.open_dataset(datadir+"z10.era5.197906.nc")

In [51]:
gh_mons = xr.open_mfdataset(gh_fpath)

In [55]:
gh_mons = gh_mons.z.sel(latitude=slice(-20,-90))

In [58]:
gh_mons

<xarray.DataArray 'z' (time: 46, latitude: 141, longitude: 720)> Size: 19MB
dask.array<getitem, shape=(46, 141, 720), dtype=float32, chunksize=(1, 141, 720), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 368B 1979-06-01 1980-06-01 ... 2024-06-01
  * latitude   (latitude) float32 564B -20.0 -20.5 -21.0 ... -89.0 -89.5 -90.0
  * longitude  (longitude) float32 3kB -180.0 -179.5 -179.0 ... 179.0 179.5
Attributes:
    level:                   10
    units:                   m**2 s**-2
    long_name:               Geopotential
    standard_name:           geopotential
    _FillValue_original:     -32767
    missing_value_original:  -32767

In [71]:
w

<xarray.DataArray 'latitude' (latitude: 141)> Size: 564B
array([0.96937746, 0.96781826, 0.96621966, 0.96458155, 0.96290386,
       0.9611865 , 0.95942944, 0.95763254, 0.9557957 , 0.95391893,
       0.952002  , 0.9500449 , 0.9480475 , 0.9460097 , 0.9439314 ,
       0.9418125 , 0.9396529 , 0.9374525 , 0.93521106, 0.93292856,
       0.9306049 , 0.9282398 , 0.9258333 , 0.92338514, 0.9208953 ,
       0.91836345, 0.9157896 , 0.9131735 , 0.910515  , 0.90781397,
       0.9050702 , 0.9022835 , 0.8994537 , 0.8965807 , 0.8936641 ,
       0.89070386, 0.8876997 , 0.8846514 , 0.88155884, 0.8784216 ,
       0.8752397 , 0.8720126 , 0.86874026, 0.8654223 , 0.86205846,
       0.85864854, 0.8551922 , 0.8516891 , 0.848139  , 0.84454155,
       0.8408964 , 0.83720326, 0.8334617 , 0.8296713 , 0.8258319 ,
       0.8219429 , 0.818004  , 0.8140148 , 0.80997473, 0.8058834 ,
       0.80174035, 0.7975451 , 0.7932972 , 0.788996  , 0.784641  ,
       0.78023165, 0.7757673 , 0.77124757, 0.76667154, 0.7620387 ,
       0.7573483 , 0.75259966, 0.747792  , 0.74292463, 0.73799664,
       0.7330072 , 0.7279555 , 0.72284067, 0.71766156, 0.7124173 ,
       0.70710677, 0.701729  , 0.6962827 , 0.6907668 , 0.68518   ,
       0.67952085, 0.6737882 , 0.6679804 , 0.662096  , 0.6561335 ,
       0.650091  , 0.64396685, 0.63775903, 0.63146585, 0.625085  ,
       0.61861414, 0.6120512 , 0.6053934 , 0.5986384 , 0.5917832 ,
       0.5848249 , 0.5777603 , 0.5705858 , 0.56329805, 0.55589294,
       0.5483665 , 0.54071414, 0.5329309 , 0.52501184, 0.516951  ,
       0.50874263, 0.50037986, 0.49185556, 0.4831619 , 0.47429004,
       0.46523076, 0.4559733 , 0.44650638, 0.43681696, 0.42689052,
       0.4167112 , 0.40626046, 0.395518  , 0.38445997, 0.37305912,
       0.36128417, 0.34909788, 0.3364569 , 0.32330856, 0.30958965,
       0.2952216 , 0.2801055 , 0.26411456, 0.24707997, 0.22877057,
       0.20885271, 0.18681405, 0.16179307, 0.13210747, 0.09341606,
              nan], dtype=float32)
Coordinates:
  * latitude  (latitude) float32 564B -20.0 -20.5 -21.0 ... -89.0 -89.5 -90.0
Attributes:
    long_name:  latitude
    units:      degrees_north

In [ ]:
gh_mons = gh_mons.isel(latitude=slice(None, None, -1))

In [89]:
gh_mons['latitude'].values

array([-20. , -20.5, -21. , -21.5, -22. , -22.5, -23. , -23.5, -24. ,
       -24.5, -25. , -25.5, -26. , -26.5, -27. , -27.5, -28. , -28.5,
       -29. , -29.5, -30. , -30.5, -31. , -31.5, -32. , -32.5, -33. ,
       -33.5, -34. , -34.5, -35. , -35.5, -36. , -36.5, -37. , -37.5,
       -38. , -38.5, -39. , -39.5, -40. , -40.5, -41. , -41.5, -42. ,
       -42.5, -43. , -43.5, -44. , -44.5, -45. , -45.5, -46. , -46.5,
       -47. , -47.5, -48. , -48.5, -49. , -49.5, -50. , -50.5, -51. ,
       -51.5, -52. , -52.5, -53. , -53.5, -54. , -54.5, -55. , -55.5,
       -56. , -56.5, -57. , -57.5, -58. , -58.5, -59. , -59.5, -60. ,
       -60.5, -61. , -61.5, -62. , -62.5, -63. , -63.5, -64. , -64.5,
       -65. , -65.5, -66. , -66.5, -67. , -67.5, -68. , -68.5, -69. ,
       -69.5, -70. , -70.5, -71. , -71.5, -72. , -72.5, -73. , -73.5,
       -74. , -74.5, -75. , -75.5, -76. , -76.5, -77. , -77.5, -78. ,
       -78.5, -79. , -79.5, -80. , -80.5, -81. , -81.5, -82. , -82.5,
       -83. , -83.5,

In [90]:
np.deg2rad(gh_mons['latitude'].values)

array([-0.34906584, -0.3577925 , -0.36651915, -0.37524578, -0.38397244,
       -0.3926991 , -0.40142572, -0.41015238, -0.41887903, -0.42760566,
       -0.43633232, -0.44505894, -0.4537856 , -0.46251225, -0.47123888,
       -0.47996554, -0.4886922 , -0.49741882, -0.5061455 , -0.51487213,
       -0.5235988 , -0.53232545, -0.54105204, -0.5497787 , -0.55850536,
       -0.567232  , -0.57595867, -0.58468527, -0.5934119 , -0.6021386 ,
       -0.61086524, -0.6195919 , -0.62831855, -0.63704515, -0.6457718 ,
       -0.65449846, -0.6632251 , -0.6719518 , -0.6806784 , -0.689405  ,
       -0.6981317 , -0.70685834, -0.715585  , -0.72431165, -0.7330383 ,
       -0.7417649 , -0.75049156, -0.7592182 , -0.7679449 , -0.7766715 ,
       -0.7853982 , -0.7941248 , -0.80285144, -0.8115781 , -0.82030475,
       -0.8290314 , -0.83775806, -0.84648466, -0.8552113 , -0.863938  ,
       -0.87266463, -0.8813913 , -0.8901179 , -0.89884454, -0.9075712 ,
       -0.91629785, -0.9250245 , -0.93375117, -0.94247776, -0.95

In [97]:
np.sqrt(np.cos(np.deg2rad(gh_mons['latitude'].values))) 

/tmp/ipykernel_10925/3910112419.py:1: RuntimeWarning: invalid value encountered in sqrt
  np.sqrt(np.cos(np.deg2rad(gh_mons['latitude'].values)))


array([0.96937746, 0.96781826, 0.96621966, 0.96458155, 0.96290386,
       0.9611865 , 0.95942944, 0.95763254, 0.9557957 , 0.95391893,
       0.952002  , 0.9500449 , 0.9480475 , 0.9460097 , 0.9439314 ,
       0.9418125 , 0.9396529 , 0.9374525 , 0.93521106, 0.93292856,
       0.9306049 , 0.9282398 , 0.9258333 , 0.92338514, 0.9208953 ,
       0.91836345, 0.9157896 , 0.9131735 , 0.910515  , 0.90781397,
       0.9050702 , 0.9022835 , 0.8994537 , 0.8965807 , 0.8936641 ,
       0.89070386, 0.8876997 , 0.8846514 , 0.88155884, 0.8784216 ,
       0.8752397 , 0.8720126 , 0.86874026, 0.8654223 , 0.86205846,
       0.85864854, 0.8551922 , 0.8516891 , 0.848139  , 0.84454155,
       0.8408964 , 0.83720326, 0.8334617 , 0.8296713 , 0.8258319 ,
       0.8219429 , 0.818004  , 0.8140148 , 0.80997473, 0.8058834 ,
       0.80174035, 0.7975451 , 0.7932972 , 0.788996  , 0.784641  ,
       0.78023165, 0.7757673 , 0.77124757, 0.76667154, 0.7620387 ,
       0.7573483 , 0.75259966, 0.747792  , 0.74292463, 0.73799

In [103]:
ds_sub

<xarray.DataArray 'z' (time: 46, latitude: 36, longitude: 180)> Size: 1MB
dask.array<getitem, shape=(46, 36, 180), dtype=float32, chunksize=(1, 36, 180), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 368B 1979-06-01 1980-06-01 ... 2024-06-01
  * latitude   (latitude) float32 144B -20.0 -22.0 -24.0 ... -86.0 -88.0 -90.0
  * longitude  (longitude) float32 720B -180.0 -178.0 -176.0 ... 176.0 178.0
Attributes:
    level:                   10
    units:                   m**2 s**-2
    long_name:               Geopotential
    standard_name:           geopotential
    _FillValue_original:     -32767
    missing_value_original:  -32767

In [104]:
w

array([0.96937746, 0.96290386, 0.9557957 , 0.9480475 , 0.9396529 ,
       0.9306049 , 0.9208953 , 0.910515  , 0.8994537 , 0.8876997 ,
       0.8752397 , 0.86205846, 0.848139  , 0.8334617 , 0.818004  ,
       0.80174035, 0.784641  , 0.76667154, 0.747792  , 0.7279555 ,
       0.70710677, 0.68518   , 0.662096  , 0.63775903, 0.6120512 ,
       0.5848249 , 0.55589294, 0.52501184, 0.49185556, 0.4559733 ,
       0.4167112 , 0.37305912, 0.32330856, 0.26411456, 0.18681405,
              nan], dtype=float32)

In [102]:
ds_sub = gh_mons.isel(latitude=slice(0, None, 4), longitude=slice(0, None, 4))

# convert z (m^2/s^2) -> meters and apply weights AFTER the averaging
w = np.sqrt(np.cos(np.deg2rad(ds_sub['latitude'].values)))  # √cos(lat), 1D
pwt_mon = (ds_sub / 9.8)* w                  # weighted monthly fields

# anomalies across years for this month
clim = pwt_mon.mean('time')
pwt_anom = pwt_mon - clim

# EOF input: (time, lat, lon)
da_mon = pwt_mon.transpose('time', 'latitude', 'longitude').astype('float32').compute()


/tmp/ipykernel_10925/235489034.py:4: RuntimeWarning: invalid value encountered in sqrt
  w = np.sqrt(np.cos(np.deg2rad(ds_sub['latitude'].values)))  # √cos(lat), 1D


ValueError: operands could not be broadcast together with shapes (46, 36, 180) (36,)

In [99]:
gh_mons['latitude']

<xarray.DataArray 'latitude' (latitude: 141)> Size: 564B
array([-20. , -20.5, -21. , -21.5, -22. , -22.5, -23. , -23.5, -24. , -24.5,
       -25. , -25.5, -26. , -26.5, -27. , -27.5, -28. , -28.5, -29. , -29.5,
       -30. , -30.5, -31. , -31.5, -32. , -32.5, -33. , -33.5, -34. , -34.5,
       -35. , -35.5, -36. , -36.5, -37. , -37.5, -38. , -38.5, -39. , -39.5,
       -40. , -40.5, -41. , -41.5, -42. , -42.5, -43. , -43.5, -44. , -44.5,
       -45. , -45.5, -46. , -46.5, -47. , -47.5, -48. , -48.5, -49. , -49.5,
       -50. , -50.5, -51. , -51.5, -52. , -52.5, -53. , -53.5, -54. , -54.5,
       -55. , -55.5, -56. , -56.5, -57. , -57.5, -58. , -58.5, -59. , -59.5,
       -60. , -60.5, -61. , -61.5, -62. , -62.5, -63. , -63.5, -64. , -64.5,
       -65. , -65.5, -66. , -66.5, -67. , -67.5, -68. , -68.5, -69. , -69.5,
       -70. , -70.5, -71. , -71.5, -72. , -72.5, -73. , -73.5, -74. , -74.5,
       -75. , -75.5, -76. , -76.5, -77. , -77.5, -78. , -78.5, -79. , -79.5,
       -80. , -80.5, -81. , -81.5, -82. , -82.5, -83. , -83.5, -84. , -84.5,
       -85. , -85.5, -86. , -86.5, -87. , -87.5, -88. , -88.5, -89. , -89.5,
       -90. ], dtype=float32)
Coordinates:
  * latitude  (latitude) float32 564B -20.0 -20.5 -21.0 ... -89.0 -89.5 -90.0
Attributes:
    long_name:  latitude
    units:      degrees_north

In [100]:
eof(da_mon)

LinAlgError: SVD did not converge

In [107]:
gh_dir

PosixPath('/net/cfc/s2s/shared_data/Datasets/ERA5/plev')

In [108]:
xr.open_dataset("data/z10_eunpa/" + "daily.z10.198406.nc")

<xarray.Dataset> Size: 3MB
Dimensions:    (time: 30, latitude: 71, longitude: 360)
Coordinates:
  * time       (time) datetime64[ns] 240B 1984-06-01T11:00:00 ... 1984-06-30T...
  * latitude   (latitude) float32 284B -90.0 -89.0 -88.0 ... -22.0 -21.0 -20.0
  * longitude  (longitude) float32 1kB -180.0 -179.0 -178.0 ... 178.0 179.0
Data variables:
    z          (time, latitude, longitude) float32 3MB ...